# Advanced Problems with Solutions: Python Compile-Time Optimizations

These problems focus on what many developers casually call "peephole optimizations": CPython optimizations that happen while source code is compiled to bytecode. In modern CPython, some historically "peephole" behavior, especially constant folding, is handled by the AST optimizer rather than only by a bytecode peephole pass.

**Audience:** advanced Python learners who can already read simple bytecode and understand mutability, hashing, and benchmarking caveats.

**How to use this notebook:** try each problem before opening the solution section. Run the code in Python 3.13 for the most consistent results.


In [1]:
import dis
import sys
import timeit
from types import CodeType

print(sys.version)


3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]


## Helper utilities

The helpers below make the exercises easier to inspect. They avoid relying on a single bytecode instruction name where possible, because CPython bytecode can change between versions.


In [2]:
def show_consts(fn):
    """Return the constants stored on a function's code object."""
    return fn.__code__.co_consts

def show_names(fn):
    """Return names referenced by a function's code object."""
    return fn.__code__.co_names

def disassemble(fn):
    """Print a readable disassembly."""
    print(f"Disassembly for {fn.__name__}:")
    dis.dis(fn)

def contains_const(fn, value):
    """True if value is present in co_consts using equality."""
    return any(c == value for c in fn.__code__.co_consts)

def contains_type(fn, typ):
    """True if any constant has exactly the specified type."""
    return any(type(c) is typ for c in fn.__code__.co_consts)


## Problem 1 — Predict constant folding boundaries

Consider the function below.

**Tasks**

1. Predict which assigned values are stored as precomputed constants.
2. Explain why each expression is or is not safe to fold.
3. Verify using `co_consts` and `dis`.


In [3]:
def problem_1():
    a = 2 ** 10
    b = 24 * 60 * 60
    c = ("py", "thon") * 3
    d = "ab" * 20
    e = "ab" * 5000
    f = [1, 2, 3] * 3
    g = {"x": 1, "y": 2}
    h = (1, 2, [3, 4])
    return a, b, c, d, e, f, g, h

show_consts(problem_1)


(None,
 1024,
 86400,
 ('py', 'thon', 'py', 'thon', 'py', 'thon'),
 'abababababababababababababababababababab',
 'ab',
 5000,
 (1, 2, 3),
 3,
 1,
 2,
 ('x', 'y'),
 4)

In [4]:
disassemble(problem_1)


Disassembly for problem_1:
  1           RESUME                   0

  2           LOAD_CONST               1 (1024)
              STORE_FAST               0 (a)

  3           LOAD_CONST               2 (86400)
              STORE_FAST               1 (b)

  4           LOAD_CONST               3 (('py', 'thon', 'py', 'thon', 'py', 'thon'))
              STORE_FAST               2 (c)

  5           LOAD_CONST               4 ('abababababababababababababababababababab')
              STORE_FAST               3 (d)

  6           LOAD_CONST               5 ('ab')
              LOAD_CONST               6 (5000)
              BINARY_OP                5 (*)
              STORE_FAST               4 (e)

  7           BUILD_LIST               0
              LOAD_CONST               7 ((1, 2, 3))
              LIST_EXTEND              1
              LOAD_CONST               8 (3)
              BINARY_OP                5 (*)
              STORE_FAST               5 (f)

  8           LOAD_C

### Solution 1

`a`, `b`, `c`, and usually `d` are compile-time candidates because they are built entirely from immutable constants and pure constant operations.

`f` is not folded into a single constant list because lists are mutable. Reusing a prebuilt list object would create observable aliasing bugs.

`g` is not stored as a single constant dictionary for the same mutability reason. The keys and values may appear in `co_consts`, but the dictionary object itself is built at runtime.

`h` is not a deeply immutable constant because it contains a list. The tuple object cannot safely be treated as a constant object when one of its elements is mutable.

`e` demonstrates an implementation limit: CPython intentionally avoids folding arbitrarily large constants. Exact limits are implementation details, so the robust lesson is: do not write correctness-sensitive code that depends on a particular folding threshold.


In [5]:
assert contains_const(problem_1, 1024)
assert contains_const(problem_1, 86400)
assert contains_const(problem_1, ("py", "thon", "py", "thon", "py", "thon"))

# These checks express semantic expectations without depending on every bytecode detail.
assert not contains_type(problem_1, list)
assert not contains_type(problem_1, dict)

show_consts(problem_1)


(None,
 1024,
 86400,
 ('py', 'thon', 'py', 'thon', 'py', 'thon'),
 'abababababababababababababababababababab',
 'ab',
 5000,
 (1, 2, 3),
 3,
 1,
 2,
 ('x', 'y'),
 4)

## Problem 2 — Prove that mutable constants are not shared accidentally

A dangerous optimizer might prebuild `[0] * 3` once and reuse it on every call. CPython must not do that.

**Tasks**

1. Write a function that creates a list literal or repeated list.
2. Mutate the returned result from one call.
3. Prove a later call is unaffected.
4. Explain why this matters for optimizer correctness.


In [6]:
def problem_2():
    return [0] * 3

first = problem_2()
first.append(99)

second = problem_2()

first, second, first is second


([0, 0, 0, 99], [0, 0, 0], False)

### Solution 2

The two calls return different list objects. If the compiler stored the list as a reusable constant, mutation through one call would affect future calls. That would change Python's semantics, so mutable containers must be constructed at runtime.

The optimizer may still store immutable pieces such as integers in `co_consts`, but it cannot replace the whole list expression with a shared list object.


In [7]:
assert first == [0, 0, 0, 99]
assert second == [0, 0, 0]
assert first is not second
assert not contains_type(problem_2, list)

show_consts(problem_2), dis.Bytecode(problem_2).dis()


((None, 0, 3),
 '  1           RESUME                   0\n\n  2           LOAD_CONST               1 (0)\n              BUILD_LIST               1\n              LOAD_CONST               2 (3)\n              BINARY_OP                5 (*)\n              RETURN_VALUE\n')